# Text-to-SVG V4: QLoRA Fine-Tuning + Inference

**NYU Deep Learning Spring 2026 — Kaggle Competition**

**Environment:** Google Colab Pro (A100 GPU) + Google Drive

### Setup Instructions
1. Upload `train.csv` and `test.csv` to your Google Drive under `MyDrive/svg-competition/`
2. Set Runtime → Change runtime type → **GPU** (A100 if available)
3. Run all cells in order
4. Adapter weights will be saved to `MyDrive/svg-competition/svg-lora-adapter/`

### V4 key fixes
- **Fixed xmlns normalizer bug** — V3 rejected 99.4% of training data due to duplicate xmlns. Now gets ~48K clean samples.
- **Tag-whitelisted training data** — model only trains on SVGs with allowed tags. No more disallowed tag output.
- **256×256 canvas normalization** via XML parse (not regex) — proper structure.
- **Qwen3.5-2B** via Unsloth (training) + text-only Qwen (inference) to avoid vision model issue.
- **3 epochs** — clean data is worth more passes.
- **lora_r=64** — more expressive with clean data.
- **Consistent short system prompt** for train and inference.
- **Strict validation** on every output — guaranteed valid submission.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Contents: {os.listdir(PROJECT_DIR)}')

In [ ]:
!pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

In [ ]:
import os, re, time, random, json
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Configuration

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'

SYSTEM_PROMPT = 'Generate SVG code.'

ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter'
}

CONFIG = {
    # ── Model ──
    'model_name': 'unsloth/Qwen3.5-2B',
    'inference_model': 'Qwen/Qwen3.5-2B',       # Text-only for inference
    'max_seq_length': 2048,

    # ── LoRA ──
    'lora_r': 64,                                # Higher rank — clean data is worth it
    'lora_alpha': 128,                           # alpha/r = 2.0

    # ── Training ──
    'learning_rate': 2e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 16,
    'gradient_accumulation_steps': 1,             # Effective batch = 16
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'max_grad_norm': 0.3,

    # ── Logging ──
    'logging_steps': 20,
    'save_steps': 500,
    'eval_steps': 500,
    'save_total_limit': 3,
    'output_dir': '/content/svg-lora-checkpoints',

    # ── Data ──
    'train_csv': f'{PROJECT_DIR}/train.csv',
    'test_csv': f'{PROJECT_DIR}/test.csv',
    'eval_fraction': 0.02,
    'max_svg_chars': 16000,

    # ── Output ──
    'adapter_save_dir': f'{PROJECT_DIR}/svg-lora-adapter-v4',
}

for key in ['train_csv', 'test_csv']:
    print(f'{key}: {"FOUND" if os.path.exists(CONFIG[key]) else "NOT FOUND ⚠️"}')

## 3. Load, Normalize & Clean Training Data

**V4 fix:** The V3 normalizer had `root.set('xmlns', ...)` which duplicated the xmlns attribute,
causing `ET.fromstring()` to reject 99.4% of training SVGs on re-parse.

Fix: let `ET.register_namespace` handle xmlns, only add it if missing after serialization.

In [ ]:
def normalize_svg_to_256(svg_text):
    """Parse SVG, validate tags, set 256x256 canvas, re-serialize cleanly."""
    try:
        ET.register_namespace('', 'http://www.w3.org/2000/svg')
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return None

    root_tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
    if root_tag != 'svg':
        return None

    # Validate all tags against competition whitelist
    path_count = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag not in ALLOWED_TAGS:
            return None
        if tag == 'path':
            path_count += 1
    if path_count > 256:
        return None

    # Set canvas — do NOT set xmlns manually (causes duplicate attribute bug)
    root.set('width', '256')
    root.set('height', '256')
    if 'viewBox' not in root.attrib:
        root.set('viewBox', '0 0 256 256')

    svg = ET.tostring(root, encoding='unicode')
    svg = svg.replace('ns0:', '').replace(':ns0', '')

    # Add xmlns only if ET didn't include it
    if 'xmlns=' not in svg:
        svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)

    # Verify clean re-parse
    try:
        ET.fromstring(svg)
    except ET.ParseError:
        return None

    if len(svg) > CONFIG['max_svg_chars']:
        return None

    return svg

print('Normalizer ready (xmlns bug fixed).')

In [ ]:
df = pd.read_csv(CONFIG['train_csv'])
print(f'Raw dataset: {len(df)} rows')

valid_rows = []
reject_reasons = Counter()

for _, row in df.iterrows():
    svg = str(row['svg']).strip()
    prompt = str(row['prompt']).strip()

    if not prompt or len(prompt) < 5:
        reject_reasons['bad_prompt'] += 1
        continue

    normalized = normalize_svg_to_256(svg)
    if normalized is None:
        reject_reasons['normalization_failed'] += 1
        continue

    # Collapse whitespace for compact training
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    valid_rows.append({'prompt': prompt, 'svg': normalized})

print(f'Clean: {len(valid_rows)} / {len(df)} ({100*len(valid_rows)/len(df):.1f}%)')
for r, c in reject_reasons.most_common():
    print(f'  {r}: {c}')

# Verify
sample = valid_rows[0]
assert 'width="256"' in sample['svg'], 'Canvas not set!'
assert sample['svg'].count('xmlns=') == 1, f'Duplicate xmlns! count={sample["svg"].count("xmlns=")}'
print(f'\nSample ({len(sample["svg"])} chars): {sample["svg"][:200]}...')
print('Canvas ✓ | Single xmlns ✓')

In [ ]:
random.shuffle(valid_rows)
n_eval = max(100, int(len(valid_rows) * CONFIG['eval_fraction']))
train_dataset = Dataset.from_list(valid_rows[n_eval:])
eval_dataset = Dataset.from_list(valid_rows[:n_eval])
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

## 4. Format for SFT

In [ ]:
def format_chat(example):
    text = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{example["prompt"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{example["svg"]}<|im_end|>'
    )
    return {'text': text}

train_formatted = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
eval_formatted = eval_dataset.map(format_chat, remove_columns=eval_dataset.column_names)

# Filter samples that won't fit in sequence length
before = len(train_formatted)
train_formatted = train_formatted.filter(lambda x: len(x['text']) < CONFIG['max_seq_length'] * 3)
print(f'Train: {before} → {len(train_formatted)} after length filter')
print(f'Eval: {len(eval_formatted)}')
print(f'\nSample: {train_formatted[0]["text"][:400]}')

## 5. Load Model via Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps', eval_steps=CONFIG['eval_steps'],
    save_strategy='steps', save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    report_to='none', optim='paged_adamw_8bit', lr_scheduler_type='cosine',
    seed=SEED,
    max_length=CONFIG['max_seq_length'], packing=True, dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train_formatted, eval_dataset=eval_formatted, args=sft_config,
)

eff_batch = CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
print(f'Effective batch: {eff_batch} | Max steps: {trainer.state.max_steps}')

In [ ]:
t0 = time.time()
train_result = trainer.train()
elapsed = (time.time() - t0) / 60
print(f'\nTraining complete in {elapsed:.1f} minutes')
print(f'Final train loss: {train_result.training_loss:.4f}')

## 7. Save Adapter

In [ ]:
adapter_dir = CONFIG['adapter_save_dir']
os.makedirs(adapter_dir, exist_ok=True)
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

summary = {
    'model': CONFIG['model_name'], 'lora_r': CONFIG['lora_r'],
    'lora_alpha': CONFIG['lora_alpha'], 'epochs': CONFIG['num_train_epochs'],
    'batch': eff_batch, 'lr': CONFIG['learning_rate'],
    'train_samples': len(train_formatted), 'final_loss': train_result.training_loss,
    'time_min': elapsed, 'gpu': torch.cuda.get_device_name(0), 'seed': SEED,
}
with open(f'{PROJECT_DIR}/training_summary_v4.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Adapter saved: {adapter_dir}')
print(f'Summary: {summary}')

## 8. Load Text-Only Model for Inference

Qwen3.5 via Unsloth loads as a vision model — `model.generate()` tries to parse inputs as images.
Fix: load `Qwen/Qwen3.5-2B` (text-only) from official Qwen and apply the trained adapter.

In [ ]:
# Free training model VRAM
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM freed: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB available')

In [ ]:
!pip install -q --upgrade transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print('Loading text-only base model...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['inference_model'])
text_tokenizer.padding_side = 'left'
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
    text_tokenizer.pad_token_id = text_tokenizer.eos_token_id

inf_model = AutoModelForCausalLM.from_pretrained(
    CONFIG['inference_model'],
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

print('Loading trained LoRA adapter...')
inf_model = PeftModel.from_pretrained(inf_model, CONFIG['adapter_save_dir'])
inf_model = inf_model.merge_and_unload()
inf_model.eval()
print(f'Inference model ready on {inf_model.device}')

## 9. Inference

In [ ]:
SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ''

def fallback_svg(prompt):
    colors = ['red','blue','green','yellow','orange','purple','black','white','pink','brown','gray']
    fill = 'gray'
    for c in colors:
        if c in prompt.lower(): fill = c; break
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        f'<rect width="256" height="256" fill="white"/>'
        f'<circle cx="128" cy="128" r="64" fill="{fill}"/>'
        '</svg>'
    )

def strict_validate(svg):
    if not svg or len(svg) > 16000: return False
    try: root = ET.fromstring(svg)
    except ET.ParseError: return False
    root_tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
    if root_tag != 'svg': return False
    pc = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag not in ALLOWED_TAGS: return False
        if tag == 'path': pc += 1
    return pc <= 256

def clean_and_validate_svg(svg):
    """Parse, remove bad tags, fix canvas, re-serialize, validate."""
    if not svg: return None
    ET.register_namespace('', 'http://www.w3.org/2000/svg')
    try: root = ET.fromstring(svg)
    except ET.ParseError: return None
    # Remove disallowed tags (keep children)
    def remove_bad(elem):
        for child in list(elem):
            tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
            if tag not in ALLOWED_TAGS: elem.remove(child)
            else: remove_bad(child)
    remove_bad(root)
    root.set('width', '256')
    root.set('height', '256')
    if 'viewBox' not in root.attrib: root.set('viewBox', '0 0 256 256')
    svg_out = ET.tostring(root, encoding='unicode')
    svg_out = svg_out.replace('ns0:', '').replace(':ns0', '')
    if 'xmlns=' not in svg_out:
        svg_out = svg_out.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    return svg_out if strict_validate(svg_out) else None

def generate_svg(prompt):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text = text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = text_tokenizer(text, return_tensors='pt').to(inf_model.device)

    with torch.no_grad():
        output_ids = inf_model.generate(
            **inputs, max_new_tokens=512,
            do_sample=True, temperature=0.5, top_p=0.9,
            repetition_penalty=1.4,
        )
    decoded = text_tokenizer.decode(output_ids[0], skip_special_tokens=False)
    assistant = decoded.split('<|im_start|>assistant\n')[-1]

    # Handle Qwen3.5 thinking block
    if '</think>' in assistant:
        assistant = assistant.split('</think>')[-1]
    for tok in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
        assistant = assistant.split(tok)[0]

    # Try 1: complete SVG
    svg = extract_svg(assistant)
    if svg:
        result = clean_and_validate_svg(svg)
        if result: return result

    # Try 2: force-close incomplete SVG
    if '<svg' in assistant:
        start = assistant.index('<svg')
        partial = assistant[start:].rstrip()
        last_sc = partial.rfind('/>')
        last_et = partial.rfind('</')
        if last_et != -1:
            try: partial = partial[:partial.index('>', last_et) + 1]
            except ValueError:
                if last_sc != -1: partial = partial[:last_sc + 2]
        elif last_sc != -1:
            partial = partial[:last_sc + 2]
        if '</svg>' not in partial: partial += '</svg>'
        result = clean_and_validate_svg(partial)
        if result: return result

    # Try 3: regex canvas fix for SVGs that fail XML parse
    svg = extract_svg(assistant)
    if not svg and '<svg' in assistant:
        start = assistant.index('<svg')
        svg = assistant[start:]
        if '</svg>' not in svg: svg += '</svg>'
    if svg:
        svg = re.sub(r"width=['\"][^'\"]*['\"]", 'width="256"', svg, count=1)
        svg = re.sub(r"height=['\"][^'\"]*['\"]", 'height="256"', svg, count=1)
        if 'viewBox' not in svg:
            svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
        if 'xmlns' not in svg:
            svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
        svg = svg.replace('""', '"')
        if len(svg) <= 16000 and svg.strip().startswith('<svg') and '</svg>' in svg:
            return svg

    return fallback_svg(prompt)

# Quick test
print('--- Inference test ---')
for p in ['a red circle', 'blue star icon', 'green tree with brown trunk']:
    t1 = time.time()
    svg = generate_svg(p)
    valid = strict_validate(svg)
    fb = len(svg) < 190
    print(f'{time.time()-t1:.1f}s | len={len(svg)} | valid={valid} | fb={fb} | "{p}"')
    if not fb:
        print(f'  {svg[:150]}...')

In [ ]:
# ── Generate all 1000 SVGs ──
test_df = pd.read_csv(CONFIG['test_csv'])
print(f'Test: {len(test_df)} rows')

rows = []
fallback_count = 0
t0 = time.time()

for idx, row in test_df.iterrows():
    prompt = str(row['prompt']).strip()
    t1 = time.time()
    svg = generate_svg(prompt)
    gen_time = time.time() - t1
    is_fb = len(svg) < 190
    if is_fb: fallback_count += 1
    rows.append({'id': row['id'], 'svg': svg})
    print(f'  [{idx+1}/{len(test_df)}] {gen_time:.1f}s | len={len(svg)} | fb={is_fb} | total_fb={fallback_count}')

elapsed_total = (time.time() - t0) / 60
print(f'\nDone! {len(rows)} SVGs in {elapsed_total:.1f} min')
print(f'Fallbacks: {fallback_count}/{len(rows)} ({100*fallback_count/len(rows):.1f}%)')

In [ ]:
# ── Save submission ──
sub_df = pd.DataFrame(rows)
svg_lengths = sub_df['svg'].str.len()
print(f'SVG lengths — mean: {svg_lengths.mean():.0f}, max: {svg_lengths.max():.0f}')

SUBMISSION_PATH = f'{PROJECT_DIR}/submission_v4.csv'
sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved: {SUBMISSION_PATH}')

from google.colab import files
files.download(SUBMISSION_PATH)

## AI Tooling Disclosure

- **Claude (Anthropic)**: Coding assistance, debugging.